# nb14: Effective Mass + Janus Asymmetry Descriptors

Two new descriptor families:
1. **Effective mass at VBM/CBM** from vasprun.xml band structure (parabolic fit)
2. **Janus / out-of-plane structural asymmetry** from POSCAR_std (z-dipoles, layer composition)

Both are exact (not proxies). Both target Rashba physics directly.


In [1]:
import os, glob, warnings, numpy as np, pandas as pd
from time import time
from itertools import combinations
from urllib.parse import unquote

from pymatgen.core import Structure
from pymatgen.io.vasp.outputs import Vasprun

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

BASE_DIR = os.path.abspath(os.path.join('..'))
OLD_CSV = os.path.join(BASE_DIR, 'k-path', 'old+new_nb6', 'rashba_206_all_descriptors_old.csv')
NEW_CSV = os.path.join(BASE_DIR, 'k-path', 'old+new_nb6', 'rashba_206_all_descriptors.csv')
POSCAR_DIR = os.path.join(BASE_DIR, 'Inverse-design', 'rashba')
RESULTS_DIR = os.path.join('.', 'nb14_effmass_janus-results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup OK. Results dir:', RESULTS_DIR)


Setup OK. Results dir: .\nb14_effmass_janus-results


## Cell 2: Load merged data, baseline (sanity check)

In [2]:
df_old = pd.read_csv(OLD_CSV)
df_new = pd.read_csv(NEW_CSV)
ID_COLS = ['Formula', 'uid', 'kpath', 'Rashba_parameter']
TARGET = 'Rashba_parameter'

df_merged = df_old[ID_COLS].copy()
old_features = [c for c in df_old.columns if c not in ID_COLS]
new_features = [c for c in df_new.columns if c not in ID_COLS]
overlap = set(old_features) & set(new_features)
for col in old_features:
    df_merged[f'old_{col}' if col in overlap else col] = df_old[col].values
for col in new_features:
    df_merged[f'new_{col}' if col in overlap else col] = df_new[col].values

idx_max = df_merged.groupby('uid')[TARGET].idxmax()
df_99 = df_merged.loc[idx_max].reset_index(drop=True)
y_99 = df_99[TARGET].values

XGB_REG_PARAMS = dict(n_estimators=100, max_depth=3, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=1.0,
    random_state=42, verbosity=0)
BASELINE_RAW = ['E_pfrac_VBM', 'E_pfrac_CBM', 'radius_mean',
                'pmid_afs_gauss_std', 'kpath_angle_deg', 'ehull']

def resolve(name, cols):
    if name in cols: return name
    if f'old_{name}' in cols: return f'old_{name}'
    if f'new_{name}' in cols: return f'new_{name}'
    raise KeyError(name)

BASELINE = [resolve(n, df_99.columns) for n in BASELINE_RAW]

def eval_reg_99(features, df, y, seed=42):
    X = df[features].fillna(0).values
    params = dict(XGB_REG_PARAMS); params['random_state'] = seed
    model = XGBRegressor(**params)
    y_pred = cross_val_predict(model, X, y, cv=LeaveOneOut())
    return r2_score(y, y_pred), mean_absolute_error(y, y_pred)

r2_base, mae_base = eval_reg_99(BASELINE, df_99, y_99)
print(f'Baseline (C6) LOO R2 = {r2_base:.4f}, MAE = {mae_base:.4f}')
if abs(r2_base - 0.643) > 0.05:
    print('  WARNING baseline differs from nb7 by > 0.05')


Baseline (C6) LOO R2 = 0.6434, MAE = 0.4287


## Cell 3: Janus / out-of-plane asymmetry features (POSCAR_std)

For each compound, compute geometric quantities that capture inversion-symmetry breaking perpendicular to the 2D layer.

**Why z-dipole matters:** in 2D Rashba, the field gradient that drives SOC splitting comes from charge redistribution perpendicular to the layer. If atoms are arranged symmetrically about the midplane in z, no field. If heavy atoms are biased toward one face (Janus structure), large field → strong Rashba.

**z-dipole weighted by Z:** signed sum of (z_i − z_mean) × Z_i, normalized. Zero if heavy atoms are symmetric about the midplane. Non-zero if biased to one side.

In [3]:
def load_structure(uid):
    matches = glob.glob(os.path.join(POSCAR_DIR, f'*-{uid}', 'POSCAR_std'))
    if not matches:
        return None
    try:
        return Structure.from_file(matches[0])
    except Exception:
        return None


JANUS_KEYS = [
    'janus_z_range', 'janus_z_std', 'janus_n_layers',
    'janus_z_dipole_Z', 'janus_z_dipole_X', 'janus_z_dipole_mass',
    'janus_z_dipole_Z_abs', 'janus_z_dipole_X_abs', 'janus_z_dipole_mass_abs',
    'janus_top_bot_meanZ', 'janus_top_bot_meanX',
    'janus_top_bot_meanZ_abs', 'janus_top_bot_meanX_abs',
]


def janus_features(struct):
    if struct is None:
        return {k: np.nan for k in JANUS_KEYS}

    zs = struct.cart_coords[:, 2]
    z_center = zs.mean()
    z_centered = zs - z_center

    Zs = np.array([s.specie.Z for s in struct], dtype=float)
    Xs = np.array([float(s.specie.X) if s.specie.X is not None else 0.0
                   for s in struct], dtype=float)
    masses = np.array([float(s.specie.atomic_mass) for s in struct], dtype=float)

    feats = {}
    feats['janus_z_range'] = float(zs.max() - zs.min())
    feats['janus_z_std'] = float(zs.std())

    # Number of distinct z-layers (cluster within 0.5 A)
    sorted_z = np.sort(zs)
    n_layers = 1
    for i in range(1, len(sorted_z)):
        if sorted_z[i] - sorted_z[i - 1] > 0.5:
            n_layers += 1
    feats['janus_n_layers'] = n_layers

    # Weighted z-dipoles: signed quantities
    feats['janus_z_dipole_Z'] = float((z_centered * Zs).sum() / max(Zs.sum(), 1e-9))
    feats['janus_z_dipole_X'] = float((z_centered * Xs).sum() / max(Xs.sum(), 1e-9))
    feats['janus_z_dipole_mass'] = float((z_centered * masses).sum() / max(masses.sum(), 1e-9))

    # Magnitudes (the model can use either sign or magnitude)
    feats['janus_z_dipole_Z_abs'] = abs(feats['janus_z_dipole_Z'])
    feats['janus_z_dipole_X_abs'] = abs(feats['janus_z_dipole_X'])
    feats['janus_z_dipole_mass_abs'] = abs(feats['janus_z_dipole_mass'])

    # Top vs bottom layer composition difference
    top_mask = zs > z_center
    bot_mask = ~top_mask
    if top_mask.any() and bot_mask.any():
        feats['janus_top_bot_meanZ'] = float(Zs[top_mask].mean() - Zs[bot_mask].mean())
        feats['janus_top_bot_meanX'] = float(Xs[top_mask].mean() - Xs[bot_mask].mean())
    else:
        feats['janus_top_bot_meanZ'] = 0.0
        feats['janus_top_bot_meanX'] = 0.0
    feats['janus_top_bot_meanZ_abs'] = abs(feats['janus_top_bot_meanZ'])
    feats['janus_top_bot_meanX_abs'] = abs(feats['janus_top_bot_meanX'])
    return feats


print('Computing Janus features for 99 compounds...')
janus_rows = []
n_failed = 0
t0 = time()
for i, row in df_99.iterrows():
    struct = load_structure(row['uid'])
    if struct is None:
        n_failed += 1
    feats = janus_features(struct)
    feats['uid'] = row['uid']
    janus_rows.append(feats)

janus_df = pd.DataFrame(janus_rows)
print(f'Done in {time()-t0:.1f}s. Failed: {n_failed}/99. Features: {len(JANUS_KEYS)}')
print('\nSample (first 5 rows):')
print(janus_df.head())

# Quick sanity check: BiITe polymorphs - do they differ in Janus features?
print('\nBiITe polymorphs check (the polymorph case from nb12):')
biite_rows = df_99[df_99['Formula'] == 'BiITe']
for _, row in biite_rows.iterrows():
    j = janus_df[janus_df['uid'] == row['uid']].iloc[0]
    print(f"  uid={row['uid']}, alpha_R={row[TARGET]:.3f}, "
          f"z_dipole_Z={j['janus_z_dipole_Z']:+.4f}, top_bot_meanZ={j['janus_top_bot_meanZ']:+.3f}")

janus_df.to_csv(os.path.join(RESULTS_DIR, 'janus_features.csv'), index=False)


Computing Janus features for 99 compounds...
Done in 0.5s. Failed: 0/99. Features: 13

Sample (first 5 rows):
   janus_z_range  janus_z_std  janus_n_layers  janus_z_dipole_Z  \
0       3.251014     1.327871               3     -2.127535e-01   
1       3.155054     1.463264               2      6.991176e-07   
2       3.648358     1.489448               3      5.309072e-01   
3       3.341810     1.364102               3     -2.575141e-07   
4       3.390762     1.348078               3      2.733814e-05   

   janus_z_dipole_X  janus_z_dipole_mass  janus_z_dipole_Z_abs  \
0          0.004903            -0.232960          2.127535e-01   
1          0.000001             0.000001          6.991176e-07   
2         -0.263858             0.589010          5.309072e-01   
3         -0.000003            -0.000001          2.575141e-07   
4          0.000006             0.000030          2.733814e-05   

   janus_z_dipole_X_abs  janus_z_dipole_mass_abs  janus_top_bot_meanZ  \
0              0.

## Cell 4: Effective mass at VBM and CBM (vasprun.xml parsing)

For each compound, parse vasprun.xml to get the SOC band structure, find VBM and CBM, fit a parabola in a small window around each band edge along the k-path, and extract effective mass m\*.

**Conversion:** in atomic units, 1/m\* = (1/ℏ²) d²E/dk². Numerically, fitting E(k) = a + bk + ck² gives m\* = (ℏ²/2c). With E in eV and k in 1/Å, the prefactor is ℏ²/(2 m_e) = 3.81 eV·Å² so m\* (in m_e) = 3.81/(2c).

**Time estimate:** ~2 sec per compound for parsing vasprun.xml ≈ 3-5 min total for 99 compounds.

In [12]:
def find_vasprun(uid):
    folder_glob = glob.glob(os.path.join(POSCAR_DIR, f'*-{uid}'))
    if not folder_glob:
        return None
    folder = folder_glob[0]
    for f in os.listdir(folder):
        if unquote(f).endswith('/vasprun.xml'):
            return os.path.join(folder, f)
    return None


BS_KEYS = [
    'bs_m_eff_VBM', 'bs_m_eff_CBM', 'bs_m_eff_ratio',
    'bs_VBM_kdist_to_gamma', 'bs_CBM_kdist_to_gamma',
    'bs_VBM_at_gamma', 'bs_CBM_at_gamma',
    'bs_band_gap_at_VBM_kpt', 'bs_band_gap_at_CBM_kpt',
    'bs_VBM_curvature', 'bs_CBM_curvature',
]


def parabolic_fit_curvature(eigvals_band, kpoints_cart, kidx, window=3):
    """Fit parabola E(d) = a + b*d + c*d^2 in window around kidx along path.
    d = signed distance from kidx along path direction (1/A).
    Returns (m_star_in_me_units, c)."""
    n = eigvals_band.shape[0]
    i_lo = max(0, kidx - window)
    i_hi = min(n, kidx + window + 1)
    if i_hi - i_lo < 3:
        return np.nan, np.nan

    k0 = kpoints_cart[kidx]
    ds = np.array([np.linalg.norm(kpoints_cart[j] - k0) for j in range(i_lo, i_hi)])
    # signed distance: negative for j < kidx
    for j in range(i_lo, i_hi):
        if j < kidx:
            ds[j - i_lo] = -ds[j - i_lo]
    Es = np.array([eigvals_band[j] for j in range(i_lo, i_hi)])

    try:
        coef = np.polyfit(ds, Es, 2)
        c = float(coef[0])
        if abs(c) < 1e-9:
            return np.nan, c
        # m* in m_e units: 3.81 eV*A^2 = hbar^2/(2 m_e)
        m_star = 3.81 / (2.0 * c)
        return float(m_star), c
    except Exception:
        return np.nan, np.nan


def effmass_features(uid):
    vr_path = find_vasprun(uid)
    if vr_path is None:
        return {k: np.nan for k in BS_KEYS}, 'no vasprun.xml'

    try:
        vr = Vasprun(vr_path, parse_dos=False, parse_potcar_file=False,
                     exception_on_bad_xml=False)
    except Exception as e:
        return {k: np.nan for k in BS_KEYS}, f'parse failed: {e}'

    try:
        # Get eigenvalues + occupations directly (don't depend on bs.get_vbm/cbm)
        eig_dict = vr.eigenvalues   # {Spin: array of shape [n_kpoints, n_bands, 2]} where last dim is [energy, occupation]
        eig_arr = list(eig_dict.values())[0]   # SOC = single channel
        # Shape: [n_kpoints, n_bands, 2]; [..., 0] = energy, [..., 1] = occupation
        energies = eig_arr[:, :, 0].T   # transpose to [n_bands, n_kpoints] for consistency
        occs = eig_arr[:, :, 1].T

        # k-points
        kpoints_frac = np.array(vr.actual_kpoints)
        rec = vr.final_structure.lattice.reciprocal_lattice.matrix   # 1/A
        kpoints_cart = kpoints_frac @ rec

        # Find VBM and CBM manually
        # An occupied state has occ > 0.5; unoccupied has occ < 0.5 (for SOC, occupations are 0 or 1)
        vbm_E = -np.inf
        vbm_band = vbm_kidx = None
        cbm_E = np.inf
        cbm_band = cbm_kidx = None

        n_bands, n_k = energies.shape
        for b in range(n_bands):
            for k in range(n_k):
                E = energies[b, k]
                occ = occs[b, k]
                if occ > 0.5:   # occupied -> candidate for VBM
                    if E > vbm_E:
                        vbm_E = E; vbm_band = b; vbm_kidx = k
                else:           # unoccupied -> candidate for CBM
                    if E < cbm_E:
                        cbm_E = E; cbm_band = b; cbm_kidx = k

        if vbm_band is None or cbm_band is None:
            return {k: np.nan for k in BS_KEYS}, 'vbm/cbm not found'

        # Distances to gamma
        vbm_kdist = float(np.linalg.norm(kpoints_cart[vbm_kidx]))
        cbm_kdist = float(np.linalg.norm(kpoints_cart[cbm_kidx]))

        # Effective mass via parabolic fit
        mstar_vbm, curv_vbm = parabolic_fit_curvature(energies[vbm_band], kpoints_cart, vbm_kidx)
        mstar_cbm, curv_cbm = parabolic_fit_curvature(energies[cbm_band], kpoints_cart, cbm_kidx)

        if not np.isnan(mstar_vbm) and not np.isnan(mstar_cbm) and abs(mstar_vbm) > 1e-9:
            m_ratio = float(mstar_cbm / abs(mstar_vbm))
        else:
            m_ratio = np.nan

        gap_at_vbm_k = float(energies[cbm_band, vbm_kidx] - energies[vbm_band, vbm_kidx])
        gap_at_cbm_k = float(energies[cbm_band, cbm_kidx] - energies[vbm_band, cbm_kidx])

        return {
            'bs_m_eff_VBM': mstar_vbm,
            'bs_m_eff_CBM': mstar_cbm,
            'bs_m_eff_ratio': m_ratio,
            'bs_VBM_kdist_to_gamma': vbm_kdist,
            'bs_CBM_kdist_to_gamma': cbm_kdist,
            'bs_VBM_at_gamma': 1 if vbm_kdist < 0.05 else 0,
            'bs_CBM_at_gamma': 1 if cbm_kdist < 0.05 else 0,
            'bs_band_gap_at_VBM_kpt': gap_at_vbm_k,
            'bs_band_gap_at_CBM_kpt': gap_at_cbm_k,
            'bs_VBM_curvature': curv_vbm if not np.isnan(curv_vbm) else np.nan,
            'bs_CBM_curvature': curv_cbm if not np.isnan(curv_cbm) else np.nan,
        }, 'OK'
    except Exception as e:
        import traceback
        return {k: np.nan for k in BS_KEYS}, f'extraction failed: {type(e).__name__}: {e}'

print('Extracting effective mass + band edge features for 99 compounds...')
print('  (parsing vasprun.xml takes ~2 sec per compound)')
bs_rows = []
status_counter = {}
t0 = time()
for i, row in df_99.iterrows():
    feats, status = effmass_features(row['uid'])
    feats['uid'] = row['uid']
    feats['_status'] = status
    bs_rows.append(feats)
    status_counter[status] = status_counter.get(status, 0) + 1
    if (i + 1) % 20 == 0:
        print(f'  [{i+1}/99] elapsed {time()-t0:.0f}s')

bs_df = pd.DataFrame(bs_rows)
print(f'\nDone in {time()-t0:.0f}s.')
print(f'Status counts:')
for s, n in status_counter.items():
    print(f'  {s}: {n}')

# Drop status col before saving features
bs_features_df = bs_df.drop(columns=['_status'])
print(f'\nSample:')
print(bs_features_df.head())

# Sanity check: BiITe polymorphs again
print('\nBiITe polymorphs check (band-structure features):')
biite_rows = df_99[df_99['Formula'] == 'BiITe']
for _, row in biite_rows.iterrows():
    b = bs_df[bs_df['uid'] == row['uid']].iloc[0]
    print(f"  uid={row['uid']}, alpha_R={row[TARGET]:.3f}, "
          f"m_eff_VBM={b['bs_m_eff_VBM']:+.3f}, m_eff_CBM={b['bs_m_eff_CBM']:+.3f}, "
          f"VBM_at_gamma={b['bs_VBM_at_gamma']}")

bs_features_df.to_csv(os.path.join(RESULTS_DIR, 'effmass_features.csv'), index=False)


Extracting effective mass + band edge features for 99 compounds...
  (parsing vasprun.xml takes ~2 sec per compound)
  [20/99] elapsed 25s
  [40/99] elapsed 55s
  [60/99] elapsed 93s
  [80/99] elapsed 132s

Done in 156s.
Status counts:
  OK: 99

Sample:
   bs_m_eff_VBM  bs_m_eff_CBM  bs_m_eff_ratio  bs_VBM_kdist_to_gamma  \
0     -0.190464      0.213730        1.122155               1.288008   
1     -0.053226      0.059740        1.122379               0.546033   
2     -0.150514      0.341306        2.267595               0.000000   
3     -0.266768      0.225464        0.845167               0.319042   
4     -0.251392      0.438683        1.745017               0.321023   

   bs_CBM_kdist_to_gamma  bs_VBM_at_gamma  bs_CBM_at_gamma  \
0               1.288008                0                0   
1               0.577985                0                0   
2               1.028672                1                0   
3               0.319042                0                0   
4  

## Cell 5: Combine features with df_99

In [13]:
# Merge both feature dataframes onto df_99 by uid
df_99_ext = df_99.merge(janus_df, on='uid', how='left').merge(bs_features_df, on='uid', how='left')
print(f'Extended df_99: {df_99_ext.shape[0]} rows, {df_99_ext.shape[1]} cols')

# All new candidates from nb14
NEW_CANDIDATES = JANUS_KEYS + BS_KEYS

# Drop constant columns and all-NaN
nuniq = df_99_ext[NEW_CANDIDATES].nunique(dropna=True)
NEW_CANDIDATES = [c for c in NEW_CANDIDATES if nuniq.get(c, 0) > 1]
print(f'Active new candidates after dropping constants/all-NaN: {len(NEW_CANDIDATES)}')
for c in NEW_CANDIDATES:
    n_nan = df_99_ext[c].isna().sum()
    print(f'  {c:35s}  NaN: {n_nan}/99')


Extended df_99: 99 rows, 1417 cols
Active new candidates after dropping constants/all-NaN: 24
  janus_z_range                        NaN: 0/99
  janus_z_std                          NaN: 0/99
  janus_n_layers                       NaN: 0/99
  janus_z_dipole_Z                     NaN: 0/99
  janus_z_dipole_X                     NaN: 0/99
  janus_z_dipole_mass                  NaN: 0/99
  janus_z_dipole_Z_abs                 NaN: 0/99
  janus_z_dipole_X_abs                 NaN: 0/99
  janus_z_dipole_mass_abs              NaN: 0/99
  janus_top_bot_meanZ                  NaN: 0/99
  janus_top_bot_meanX                  NaN: 0/99
  janus_top_bot_meanZ_abs              NaN: 0/99
  janus_top_bot_meanX_abs              NaN: 0/99
  bs_m_eff_VBM                         NaN: 0/99
  bs_m_eff_CBM                         NaN: 0/99
  bs_m_eff_ratio                       NaN: 0/99
  bs_VBM_kdist_to_gamma                NaN: 0/99
  bs_CBM_kdist_to_gamma                NaN: 0/99
  bs_VBM_at_gamma       

## Cell 6: Phase A — C6 + 1

In [14]:
print('=' * 70)
print('  PHASE A: C6 + 1 (one new feature at a time)')
print('=' * 70)

phase_a = []
t0 = time()
for cand in NEW_CANDIDATES:
    feats = BASELINE + [cand]
    try:
        r2, mae = eval_reg_99(feats, df_99_ext, y_99)
        phase_a.append({'candidate': cand, 'r2': r2, 'mae': mae,
                        'delta_r2': r2 - r2_base})
    except Exception as e:
        print(f'  {cand}: FAILED -- {e}')

phase_a_df = pd.DataFrame(phase_a).sort_values('delta_r2', ascending=False).reset_index(drop=True)
print(f'\nDone in {time()-t0:.0f}s.')
print(f'Baseline: R2 = {r2_base:.4f}\n')
print('All C6 + 1 candidates (sorted by delta_r2):')
print(phase_a_df.to_string(index=False))
phase_a_df.to_csv(os.path.join(RESULTS_DIR, 'phase_a.csv'), index=False)


  PHASE A: C6 + 1 (one new feature at a time)

Done in 48s.
Baseline: R2 = 0.6434

All C6 + 1 candidates (sorted by delta_r2):
              candidate       r2      mae  delta_r2
    janus_top_bot_meanZ 0.644420 0.435388  0.001037
         janus_n_layers 0.627775 0.421828 -0.015608
        bs_VBM_at_gamma 0.625022 0.432353 -0.018361
  bs_VBM_kdist_to_gamma 0.612230 0.431293 -0.031152
    janus_top_bot_meanX 0.611792 0.437705 -0.031590
   janus_z_dipole_X_abs 0.608811 0.434196 -0.034572
        bs_CBM_at_gamma 0.607845 0.430566 -0.035537
janus_z_dipole_mass_abs 0.603806 0.441245 -0.039577
  bs_CBM_kdist_to_gamma 0.603707 0.440288 -0.039676
   janus_z_dipole_Z_abs 0.600779 0.438309 -0.042604
            janus_z_std 0.599204 0.444205 -0.044179
janus_top_bot_meanZ_abs 0.599115 0.447811 -0.044268
janus_top_bot_meanX_abs 0.596873 0.445378 -0.046510
         bs_m_eff_ratio 0.587909 0.449748 -0.055474
       janus_z_dipole_X 0.587412 0.444830 -0.055970
 bs_band_gap_at_CBM_kpt 0.586410 0.442097

## Cell 7: Phase B — C6 + 2 from top features

Take the top 5 single-feature winners. Test all C(5,2) = 10 pairs.

In [15]:
K = min(5, len(phase_a_df))
top_k = phase_a_df.head(K)['candidate'].tolist()
print(f'Top {K} from Phase A: {top_k}\n')

phase_b = []
for c1, c2 in combinations(top_k, 2):
    feats = BASELINE + [c1, c2]
    r2, mae = eval_reg_99(feats, df_99_ext, y_99)
    phase_b.append({'cand_1': c1, 'cand_2': c2, 'r2': r2, 'mae': mae,
                    'delta_r2': r2 - r2_base})

phase_b_df = pd.DataFrame(phase_b).sort_values('delta_r2', ascending=False).reset_index(drop=True)
print('All C6 + 2 pairs:')
print(phase_b_df.to_string(index=False))
phase_b_df.to_csv(os.path.join(RESULTS_DIR, 'phase_b.csv'), index=False)

best_a = phase_a_df.iloc[0]['r2']
best_b = phase_b_df.iloc[0]['r2']
print(f'\nBaseline:  R2 = {r2_base:.4f}')
print(f'Best A:    R2 = {best_a:.4f}  ({best_a - r2_base:+.4f})')
print(f'Best B:    R2 = {best_b:.4f}  ({best_b - r2_base:+.4f})')


Top 5 from Phase A: ['janus_top_bot_meanZ', 'janus_n_layers', 'bs_VBM_at_gamma', 'bs_VBM_kdist_to_gamma', 'janus_top_bot_meanX']

All C6 + 2 pairs:
               cand_1                cand_2       r2      mae  delta_r2
  janus_top_bot_meanZ        janus_n_layers 0.621535 0.453597 -0.021848
  janus_top_bot_meanZ       bs_VBM_at_gamma 0.614259 0.456070 -0.029124
  janus_top_bot_meanZ bs_VBM_kdist_to_gamma 0.611494 0.452873 -0.031889
       janus_n_layers bs_VBM_kdist_to_gamma 0.606687 0.443197 -0.036695
  janus_top_bot_meanZ   janus_top_bot_meanX 0.602796 0.451622 -0.040586
      bs_VBM_at_gamma bs_VBM_kdist_to_gamma 0.599112 0.459952 -0.044271
       janus_n_layers   janus_top_bot_meanX 0.592449 0.452245 -0.050934
      bs_VBM_at_gamma   janus_top_bot_meanX 0.590078 0.454624 -0.053304
       janus_n_layers       bs_VBM_at_gamma 0.587139 0.456241 -0.056244
bs_VBM_kdist_to_gamma   janus_top_bot_meanX 0.586211 0.460560 -0.057172

Baseline:  R2 = 0.6434
Best A:    R2 = 0.6444  (+0.0010)
Be

## Cell 8: Phase C — C6 + 3 (only if Phase B improved over Phase A)

In [16]:
if phase_b_df.iloc[0]['r2'] > phase_a_df.iloc[0]['r2']:
    phase_c = []
    for c1, c2, c3 in combinations(top_k, 3):
        feats = BASELINE + [c1, c2, c3]
        r2, mae = eval_reg_99(feats, df_99_ext, y_99)
        phase_c.append({'cand_1': c1, 'cand_2': c2, 'cand_3': c3,
                        'r2': r2, 'mae': mae, 'delta_r2': r2 - r2_base})
    phase_c_df = pd.DataFrame(phase_c).sort_values('delta_r2', ascending=False).reset_index(drop=True)
    print('Phase C (C6 + 3):')
    print(phase_c_df.head(10).to_string(index=False))
    phase_c_df.to_csv(os.path.join(RESULTS_DIR, 'phase_c.csv'), index=False)
    best_c = phase_c_df.iloc[0]['r2']
    print(f'\nBest C: R2 = {best_c:.4f}')
else:
    print('Phase B did not improve over A. Skipping Phase C.')
    phase_c_df = pd.DataFrame()


Phase B did not improve over A. Skipping Phase C.


## Cell 9: Multi-seed validation on the best feature set

If we found something promising (delta_r2 > 0.02), validate by running 10 seeds. If single-seed gain is real, multi-seed mean stays high; if it's noise, multi-seed mean collapses toward baseline.

In [17]:
# Pick the best feature set across all phases
best_extras = []
best_r2 = r2_base
best_phase = 'baseline'

for src_df, name, getter in [
    (phase_a_df, 'A', lambda r: [r['candidate']]),
    (phase_b_df, 'B', lambda r: [r['cand_1'], r['cand_2']]),
    (phase_c_df, 'C', lambda r: [r['cand_1'], r['cand_2'], r['cand_3']]),
]:
    if len(src_df) == 0:
        continue
    if src_df.iloc[0]['r2'] > best_r2:
        best_r2 = src_df.iloc[0]['r2']
        best_extras = getter(src_df.iloc[0])
        best_phase = name

if not best_extras:
    print('No phase improved over baseline. Stopping here.')
    print(f'Reporting: R2 = {r2_base:.4f} (C6 baseline)')
else:
    print(f'Best: Phase {best_phase}, R2 = {best_r2:.4f}, extras = {best_extras}')
    print(f'\nValidating with seeds 0-9...')
    new_feats = BASELINE + best_extras

    r2s_new, r2s_base_seeds = [], []
    for seed in range(10):
        r2_n, _ = eval_reg_99(new_feats, df_99_ext, y_99, seed=seed)
        r2_b, _ = eval_reg_99(BASELINE, df_99_ext, y_99, seed=seed)
        r2s_new.append(r2_n)
        r2s_base_seeds.append(r2_b)

    print(f'\n10-seed C6 baseline:  R2 = {np.mean(r2s_base_seeds):.4f} +/- {np.std(r2s_base_seeds):.4f}')
    print(f'10-seed with extras:  R2 = {np.mean(r2s_new):.4f} +/- {np.std(r2s_new):.4f}')

    diff = np.mean(r2s_new) - np.mean(r2s_base_seeds)
    pooled_std = np.sqrt(np.std(r2s_new)**2 + np.std(r2s_base_seeds)**2)
    ratio = diff / pooled_std if pooled_std > 0 else 0
    print(f'Difference: {diff:+.4f}, ratio to pooled std: {ratio:.2f}')
    print('  > 2.0 = real signal, 1-2 = marginal, < 1 = noise')

    pd.DataFrame({
        'seed': list(range(10)),
        'r2_baseline': r2s_base_seeds,
        'r2_with_extras': r2s_new,
    }).to_csv(os.path.join(RESULTS_DIR, 'multiseed_validation.csv'), index=False)


Best: Phase A, R2 = 0.6444, extras = ['janus_top_bot_meanZ']

Validating with seeds 0-9...

10-seed C6 baseline:  R2 = 0.6112 +/- 0.0141
10-seed with extras:  R2 = 0.6155 +/- 0.0092
Difference: +0.0043, ratio to pooled std: 0.26
  > 2.0 = real signal, 1-2 = marginal, < 1 = noise


## Cell 10: Polymorph check on the best feature set

Specifically: did adding these features change the predictions for the BiITe and STeW polymorphs that the C6 model couldn't distinguish?

In [10]:
if best_extras:
    new_feats = BASELINE + best_extras
    X_new = df_99_ext[new_feats].fillna(0).values
    X_base = df_99_ext[BASELINE].fillna(0).values

    model = XGBRegressor(**XGB_REG_PARAMS)
    y_pred_new = cross_val_predict(model, X_new, y_99, cv=LeaveOneOut())
    model = XGBRegressor(**XGB_REG_PARAMS)
    y_pred_base = cross_val_predict(model, X_base, y_99, cv=LeaveOneOut())

    polymorph_check_df = df_99[['uid', 'Formula', TARGET]].copy()
    polymorph_check_df['y_pred_C6'] = y_pred_base
    polymorph_check_df['y_pred_new'] = y_pred_new
    polymorph_check_df['resid_C6'] = polymorph_check_df[TARGET] - polymorph_check_df['y_pred_C6']
    polymorph_check_df['resid_new'] = polymorph_check_df[TARGET] - polymorph_check_df['y_pred_new']

    for f in ['BiITe', 'STeW']:
        rows = polymorph_check_df[polymorph_check_df['Formula'] == f]
        if len(rows) > 1:
            print(f'\n{f} polymorphs:')
            print(rows[['uid', TARGET, 'y_pred_C6', 'y_pred_new', 'resid_C6', 'resid_new']].to_string(index=False))

    polymorph_check_df.to_csv(os.path.join(RESULTS_DIR, 'polymorph_check.csv'), index=False)
else:
    print('No best feature set; skipping polymorph check.')



BiITe polymorphs:
         uid  Rashba_parameter  y_pred_C6  y_pred_new  resid_C6  resid_new
2d41b3dd1772             2.086   1.684129    1.483097  0.401871   0.602903
a84d988e38ac             0.467   1.441895    1.617977 -0.974895  -1.150977

STeW polymorphs:
         uid  Rashba_parameter  y_pred_C6  y_pred_new  resid_C6  resid_new
75ee10091f43             3.947   2.394058    2.136494  1.552942   1.810506
916afba26723             3.622   2.724936    2.835404  0.897064   0.786596


In [18]:
# Greedy forward selection from scratch on (9 old + 24 new) = 33 candidates
# k=2 exhaustive (528 pairs), then greedy adding 1 at a time up to k=8

OLD_POOL_RAW = ['E_pfrac_VBM', 'E_pfrac_CBM', 'radius_mean',
                'pmid_afs_gauss_std', 'kpath_angle_deg', 'ehull',
                'max_Z4', 'E_sfrac_VBM', 'E_sfrac_CBM']
OLD_POOL = []
for n in OLD_POOL_RAW:
    try:
        OLD_POOL.append(resolve(n, df_99_ext.columns))
    except KeyError:
        print(f'  WARNING: could not resolve {n}, skipping')

CANDIDATES = OLD_POOL + NEW_CANDIDATES
print(f'Total candidate pool: {len(CANDIDATES)} features')
print(f'  Old (from nb7 9-feature pool): {len(OLD_POOL)}')
print(f'  New (Janus + band-structure):  {len(NEW_CANDIDATES)}')

THRESHOLD = r2_base  # 0.6434
print(f'\nThreshold (will report combos with R2 > {THRESHOLD:.4f})')

import time
all_winners = []
t_start = time.time()

# ---------- k=2: exhaustive pairs ----------
print(f'\n{"="*70}')
print(f'  k=2: testing all {len(CANDIDATES) * (len(CANDIDATES)-1) // 2} pairs')
print(f'{"="*70}')

best_pair = None; best_pair_r2 = -np.inf; n_winners_k2 = 0
for i, (c1, c2) in enumerate(combinations(CANDIDATES, 2)):
    feats = [c1, c2]
    r2, mae = eval_reg_99(feats, df_99_ext, y_99)
    if r2 > best_pair_r2:
        best_pair_r2 = r2; best_pair = feats
    if r2 > THRESHOLD:
        all_winners.append({'k': 2, 'r2': r2, 'mae': mae, 'features': ' | '.join(feats)})
        n_winners_k2 += 1
    if (i + 1) % 100 == 0:
        print(f'  [{i+1}/{len(CANDIDATES)*(len(CANDIDATES)-1)//2}] best so far: {best_pair_r2:.4f}, winners: {n_winners_k2}')

print(f'\nk=2 done. Best pair: R2 = {best_pair_r2:.4f}')
print(f'  {best_pair}')
print(f'  Winners over threshold: {n_winners_k2}')

# ---------- k=3 to k=8: greedy add one feature ----------
current = list(best_pair)
current_r2 = best_pair_r2

for k in range(3, 9):
    print(f'\n{"="*70}')
    print(f'  k={k}: greedy add 1 to current best of size {len(current)}')
    print(f'{"="*70}')

    remaining = [c for c in CANDIDATES if c not in current]
    best_add = None; best_add_r2 = current_r2; best_add_mae = None
    n_winners_thisk = 0

    for i, cand in enumerate(remaining):
        feats = current + [cand]
        r2, mae = eval_reg_99(feats, df_99_ext, y_99)
        if r2 > best_add_r2:
            best_add_r2 = r2; best_add = cand; best_add_mae = mae
        if r2 > THRESHOLD:
            all_winners.append({'k': k, 'r2': r2, 'mae': mae, 'features': ' | '.join(feats)})
            n_winners_thisk += 1

    print(f'  Tested {len(remaining)} candidates')
    print(f'  Winners over threshold: {n_winners_thisk}')

    if best_add is None or best_add_r2 <= current_r2:
        print(f'  No improvement at k={k}. Stopping greedy chain.')
        print(f'  (Plateau at R2={current_r2:.4f} with {len(current)} features)')
        break

    current.append(best_add)
    current_r2 = best_add_r2
    print(f'  Best at k={k}: R2 = {current_r2:.4f} (added {best_add})')
    print(f'  Current set: {current}')

# ---------- Summary ----------
print(f'\n{"="*70}')
print(f'  ALL WINNERS (R2 > {THRESHOLD:.4f}), sorted by R2')
print(f'{"="*70}')

winners_df = pd.DataFrame(all_winners).sort_values('r2', ascending=False).reset_index(drop=True)
print(f'\nTotal winners: {len(winners_df)}')
print(f'Total time: {time.time() - t_start:.0f}s\n')

# Top 30 winners
print('Top 30 winning combinations:')
for _, row in winners_df.head(30).iterrows():
    print(f'  k={row["k"]}  R2={row["r2"]:.4f}  MAE={row["mae"]:.4f}  | {row["features"]}')

# Save full
winners_df.to_csv(os.path.join(RESULTS_DIR, 'greedy_winners.csv'), index=False)
print(f'\nSaved full list: {os.path.join(RESULTS_DIR, "greedy_winners.csv")}')

# Best at each k
print(f'\nBest at each k:')
for k in sorted(winners_df['k'].unique()):
    best_at_k = winners_df[winners_df['k'] == k].iloc[0]
    print(f'  k={k}: R2={best_at_k["r2"]:.4f}  | {best_at_k["features"]}')

Total candidate pool: 31 features
  Old (from nb7 9-feature pool): 7
  New (Janus + band-structure):  24

Threshold (will report combos with R2 > 0.6434)

  k=2: testing all 465 pairs
  [100/465] best so far: 0.3828, winners: 0
  [200/465] best so far: 0.3828, winners: 0
  [300/465] best so far: 0.3828, winners: 0
  [400/465] best so far: 0.3828, winners: 0

k=2 done. Best pair: R2 = 0.3828
  ['old_E_pfrac_CBM', 'pmid_afs_gauss_std']
  Winners over threshold: 0

  k=3: greedy add 1 to current best of size 2
  Tested 29 candidates
  Winners over threshold: 0
  Best at k=3: R2 = 0.5420 (added bs_CBM_kdist_to_gamma)
  Current set: ['old_E_pfrac_CBM', 'pmid_afs_gauss_std', 'bs_CBM_kdist_to_gamma']

  k=4: greedy add 1 to current best of size 3
  Tested 28 candidates
  Winners over threshold: 0
  No improvement at k=4. Stopping greedy chain.
  (Plateau at R2=0.5420 with 3 features)

  ALL WINNERS (R2 > 0.6434), sorted by R2


KeyError: 'r2'

In [19]:
# CORRECTED greedy: pre-seed with C6, then greedy-add NEW features only

C6_RAW = ['E_pfrac_VBM', 'E_pfrac_CBM', 'radius_mean',
          'pmid_afs_gauss_std', 'kpath_angle_deg', 'ehull']
C6 = [resolve(n, df_99_ext.columns) for n in C6_RAW]

# Verify baseline reproduces
r2_c6, mae_c6 = eval_reg_99(C6, df_99_ext, y_99)
print(f'C6 baseline: R2 = {r2_c6:.4f}  (expected 0.6434)')

THRESHOLD = r2_c6
print(f'Threshold: {THRESHOLD:.4f}\n')

# Greedy add 1 from NEW_CANDIDATES (24 features) to C6, up to size 10 (C6 + 4 max)
import time
all_winners = []
t_start = time.time()

current = list(C6)
current_r2 = r2_c6

for added in range(1, 5):  # add 1, 2, 3, 4 new features
    print(f'{"="*70}')
    print(f'  Adding feature #{added} to C6 (current size {len(current)})')
    print(f'{"="*70}')

    remaining = [c for c in NEW_CANDIDATES if c not in current]
    best_add = None; best_add_r2 = current_r2; best_add_mae = None
    n_winners_thisk = 0

    for cand in remaining:
        feats = current + [cand]
        r2, mae = eval_reg_99(feats, df_99_ext, y_99)
        if r2 > best_add_r2:
            best_add_r2 = r2; best_add = cand; best_add_mae = mae
        if r2 > THRESHOLD:
            all_winners.append({'k': len(feats), 'r2': r2, 'mae': mae,
                                'features': ' | '.join(feats),
                                'added': cand, 'added_at_step': added})
            n_winners_thisk += 1

    print(f'  Tested {len(remaining)} candidates')
    print(f'  Winners over C6 threshold: {n_winners_thisk}')

    if best_add is None or best_add_r2 <= current_r2:
        print(f'  No improvement. Greedy plateau at R2={current_r2:.4f} with {len(current)} features.')
        break

    current.append(best_add)
    current_r2 = best_add_r2
    print(f'  Added: {best_add}  -> R2 = {current_r2:.4f}  (delta {current_r2 - r2_c6:+.4f})')

# Summary
print(f'\n{"="*70}')
print(f'  WINNERS (R2 > {THRESHOLD:.4f}), sorted by R2')
print(f'{"="*70}')
winners_df = pd.DataFrame(all_winners).sort_values('r2', ascending=False).reset_index(drop=True)
print(f'Total winners: {len(winners_df)}')
print(f'Total time: {time.time() - t_start:.0f}s\n')
print(winners_df.head(30).to_string(index=False))
winners_df.to_csv(os.path.join(RESULTS_DIR, 'greedy_winners_C6_seeded.csv'), index=False)

print(f'\nGreedy chain summary:')
print(f'  C6 baseline (k=6):     R2 = {r2_c6:.4f}')
print(f'  Final (k={len(current)}):           R2 = {current_r2:.4f}')
print(f'  Final feature set:    {current}')

C6 baseline: R2 = 0.6434  (expected 0.6434)
Threshold: 0.6434

  Adding feature #1 to C6 (current size 6)
  Tested 24 candidates
  Winners over C6 threshold: 1
  Added: janus_top_bot_meanZ  -> R2 = 0.6444  (delta +0.0010)
  Adding feature #2 to C6 (current size 7)
  Tested 23 candidates
  Winners over C6 threshold: 0
  No improvement. Greedy plateau at R2=0.6444 with 7 features.

  WINNERS (R2 > 0.6434), sorted by R2
Total winners: 1
Total time: 100s

 k      r2      mae                                                                                                                         features               added  added_at_step
 7 0.64442 0.435388 old_E_pfrac_VBM | old_E_pfrac_CBM | old_radius_mean | pmid_afs_gauss_std | old_kpath_angle_deg | old_ehull | janus_top_bot_meanZ janus_top_bot_meanZ              1

Greedy chain summary:
  C6 baseline (k=6):     R2 = 0.6434
  Final (k=7):           R2 = 0.6444
  Final feature set:    ['old_E_pfrac_VBM', 'old_E_pfrac_CBM', 'old_radius_mean

In [20]:
# Feature compression: PCA + a few hand-crafted composites
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ============================================================================
# OPTION 1: PCA on the 24 new features
# ============================================================================
print('='*70)
print('  OPTION 1: PCA compression of 24 new features')
print('='*70)

X_new = df_99_ext[NEW_CANDIDATES].fillna(df_99_ext[NEW_CANDIDATES].median()).values
scaler = StandardScaler()
X_new_scaled = scaler.fit_transform(X_new)

pca = PCA(n_components=5)
X_pca = pca.fit_transform(X_new_scaled)

print(f'Variance explained by each PC: {pca.explained_variance_ratio_.round(3)}')
print(f'Cumulative:                    {np.cumsum(pca.explained_variance_ratio_).round(3)}')

# Add PCA features to df_99_ext
for i in range(5):
    df_99_ext[f'pca_new_{i+1}'] = X_pca[:, i]

PCA_FEATS = [f'pca_new_{i+1}' for i in range(5)]

# ============================================================================
# OPTION 2: Hand-crafted physical composites
# ============================================================================
print('\n' + '='*70)
print('  OPTION 2: Hand-crafted composites')
print('='*70)

eps = 1e-9
# Janus magnitude composite
df_99_ext['comp_janus_total'] = (
    df_99_ext['janus_z_dipole_Z_abs'].fillna(0) *
    df_99_ext['janus_z_dipole_X_abs'].fillna(0)
)

# Geometric mean of effective masses (single mass scale)
df_99_ext['comp_m_eff_geom'] = np.sqrt(
    df_99_ext['bs_m_eff_VBM'].abs().fillna(0) *
    df_99_ext['bs_m_eff_CBM'].abs().fillna(0)
)

# Rashba-like proxy: asymmetry * gap / mass
df_99_ext['comp_rashba_proxy'] = (
    df_99_ext['janus_z_dipole_Z_abs'].fillna(0) *
    df_99_ext['bs_band_gap_at_VBM_kpt'].fillna(0) /
    (df_99_ext['bs_m_eff_VBM'].abs().fillna(eps) + eps)
)

# k-distance times band-gap (offset Rashba momentum scale)
df_99_ext['comp_k_times_gap'] = (
    df_99_ext['bs_VBM_kdist_to_gamma'].fillna(0) *
    df_99_ext['bs_band_gap_at_VBM_kpt'].fillna(0)
)

# Product of mass with k-offset
df_99_ext['comp_k_times_m'] = (
    df_99_ext['bs_VBM_kdist_to_gamma'].fillna(0) *
    df_99_ext['bs_m_eff_VBM'].abs().fillna(0)
)

COMPOSITE_FEATS = ['comp_janus_total', 'comp_m_eff_geom', 'comp_rashba_proxy',
                   'comp_k_times_gap', 'comp_k_times_m']

print(f'Created {len(COMPOSITE_FEATS)} composites:')
for c in COMPOSITE_FEATS:
    print(f'  {c}: range [{df_99_ext[c].min():.3f}, {df_99_ext[c].max():.3f}]')

# ============================================================================
# Test C6 + 1 with PCA + composite features
# ============================================================================
print('\n' + '='*70)
print('  C6 + 1 with PCA components and composite features')
print('='*70)

ALL_NEW_COMPRESSED = PCA_FEATS + COMPOSITE_FEATS
print(f'Testing {len(ALL_NEW_COMPRESSED)} candidates (5 PCA + 5 composite)')
print(f'Baseline C6: R2 = {r2_c6:.4f}\n')

results = []
for cand in ALL_NEW_COMPRESSED:
    r2, mae = eval_reg_99(C6 + [cand], df_99_ext, y_99)
    results.append({'feature': cand, 'r2': r2, 'mae': mae,
                    'delta': r2 - r2_c6, 'type': 'PCA' if cand.startswith('pca_') else 'composite'})

res_df = pd.DataFrame(results).sort_values('delta', ascending=False).reset_index(drop=True)
print(res_df.to_string(index=False))

# ============================================================================
# Best 2 from these (compressed) on top of C6
# ============================================================================
print('\n' + '='*70)
print('  C6 + 2 (compressed features, all pairs)')
print('='*70)

pair_results = []
for c1, c2 in combinations(ALL_NEW_COMPRESSED, 2):
    r2, mae = eval_reg_99(C6 + [c1, c2], df_99_ext, y_99)
    pair_results.append({'f1': c1, 'f2': c2, 'r2': r2, 'mae': mae,
                         'delta': r2 - r2_c6})

pair_df = pd.DataFrame(pair_results).sort_values('delta', ascending=False).reset_index(drop=True)
print(pair_df.head(15).to_string(index=False))

best_single = res_df.iloc[0]
best_pair = pair_df.iloc[0]
print(f'\nBest single: {best_single["feature"]}, R2 = {best_single["r2"]:.4f}, delta = {best_single["delta"]:+.4f}')
print(f'Best pair:   {best_pair["f1"]} + {best_pair["f2"]}')
print(f'             R2 = {best_pair["r2"]:.4f}, delta = {best_pair["delta"]:+.4f}')

  OPTION 1: PCA compression of 24 new features
Variance explained by each PC: [0.306 0.127 0.121 0.1   0.093]
Cumulative:                    [0.306 0.433 0.554 0.654 0.747]

  OPTION 2: Hand-crafted composites
Created 5 composites:
  comp_janus_total: range [0.000, 0.148]
  comp_m_eff_geom: range [0.045, 1.577]
  comp_rashba_proxy: range [0.000, 5.932]
  comp_k_times_gap: range [0.000, 1.826]
  comp_k_times_m: range [0.000, 1.116]

  C6 + 1 with PCA components and composite features
Testing 10 candidates (5 PCA + 5 composite)
Baseline C6: R2 = 0.6434

          feature       r2      mae     delta      type
   comp_k_times_m 0.651431 0.407718  0.008048 composite
        pca_new_1 0.636044 0.420122 -0.007338       PCA
 comp_k_times_gap 0.617841 0.435346 -0.025542 composite
 comp_janus_total 0.604385 0.445953 -0.038998 composite
        pca_new_2 0.603027 0.433238 -0.040355       PCA
        pca_new_3 0.598705 0.439396 -0.044677       PCA
comp_rashba_proxy 0.595978 0.439547 -0.047405 comp